In [1]:
# -*- coding: utf-8 -*-
"""
════════════════════════════════════════════════════════════════════════════
Пункт 0. Микровариативность интервала PR
(запрос оппонента: уточнить погрешность детекции зубца P; построить ряды PP и
PR и сопоставить их с RR; понять, насколько чётко выделяется микровариативность
PR, вносящая расхождение между PP и RR)

ЕДИНЫЙ ФАЙЛ. Ничего импортировать из соседних файлов не нужно.

Установка:   pip install numpy scipy pyedflib matplotlib
Запуск:      python П0_микровариативность_PR_единый_код.py /путь/к/записям [--out results_p0]
Jupyter / Google Colab: выполнить ячейку целиком, затем  res = run_p0('/путь/к/записям')

Протокол пробы: около 5 мин лёжа, подъём стола до 45° без прерывания записи,
около 5 мин в положении подъёма; метки «Этап». Покой — от 30 с до (начало
подъёма − 5 с); наклон — от (конец подъёма + 15 с) до (+275 с).
Отведения: каналы 0, 1, 2 = I, II, III. Чтение записей, детекторы зубцов,
уточнение положений R и P, границы фаз и очистка рядов — те же, что в пп. 3, 4
(см. «Материалы и методы»).

Что считается
  1. Расхождение рядов PP и RR и его величина по сравнению с изменчивостью ритма.
  2. Погрешность измерения PR отдельно в покое и при наклоне (две независимые
     оценки), собственная изменчивость PR сверх погрешности и доля погрешности
     в дисперсии расхождения PP и RR (= разностей соседних PR).
  3. Медленная (скользящее среднее по 15 циклам) и быстрая составляющие PR
     внутри фаз; доля шума в каждой.
  4. Связь PR с длительностью предшествующего цикла RR: по всей записи и
     внутри фаз; значимость — по записям (критерий Уилкоксона), для отдельной
     записи — сдвиговыми суррогатами; проверка, какую часть связи по всей
     записи даёт скачок при смене позы.
  5. Изменение составляющих при наклоне; средний PR, форма и амплитуда зубца P
     по фазам; зависимость изменения среднего PR от точки отсчёта на зубце P.
  6. Блоки циклов: какой длины блока хватает, чтобы различать медленные
     изменения PR.
  7. Синтетическая проверка метода на настоящих рядах RR: связи нет; связь на
     всех медленных масштабах; связь только через медленный дрейф.
  8. Варианты расчёта: окно сглаживания, без удаления тренда, удаление
     медленного тренда (среднее по 101 циклу), лучшее отведение записи, без
     уточнения положений.
════════════════════════════════════════════════════════════════════════════
"""
!pip install numpy scipy pyedflib matplotlib
import argparse, csv, glob, json, math, os, warnings
import numpy as np
import pyedflib
from scipy import stats
from scipy.signal import butter, filtfilt, find_peaks
from scipy.ndimage import median_filter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker
warnings.filterwarnings('ignore')

# ═════════════════════════════ НАСТРОЙКИ ═════════════════════════════
LEAD_II_CHANNEL = 1
LEAD_III_CHANNEL = 2
P7_LEADS = {'0011': 1, '0012': 1, '0013': 1, '0014': 1, '0015': 1, '0016': 0, '0017': 2,
            '0018': 1, '0019': 1, '0020': 2, '0021': 1, '0022': 1, '0023': 1, '0024': 1,
            '0025': 2, '0026': 1, '0027': 1, '0028': 0, '0029': 1, '0030': 1}
ARTIFACT_MS = 250              # артефакт RR: отклонение от медианы 11 соседних больше 250 мс
OUTLIER_MAD = 4.0              # выброс PR/PP: быстрая часть дальше 4 MAD от медианы
COVERAGE = 0.8                 # фаза учитывается, если PR определён хотя бы в 80 % её циклов
SMOOTH = 15                    # медленная составляющая — скользящее среднее по 15 циклам
N_SURR = 500                   # сдвиговых суррогатов на запись и фазу
MIN_SHIFT = 30                 # минимальный сдвиг суррогата, циклов
BLOCKS = (20, 40, 80)          # длины блоков, циклов
SEED = 0


# ═════════════ ЧАСТЬ 1. Чтение, детекторы, границы фаз, уточнение, очистка ═════════════
#   (без изменений, как в пп. 3 и 4; см. «Материалы и методы»)

def read_ecg(path, ch=0):
    f = pyedflib.EdfReader(path)
    fs = float(f.getSampleFrequency(ch))
    sig = f.readSignal(ch).astype(float)
    f.close()
    return sig, fs


def detect_r(sig, fs, band=(5.0, 20.0), integ_ms=120,
             min_rr_ms=300, win_s=10.0, k=0.35, refine_ms=100):
    x = np.asarray(sig, float)
    x = x - np.median(x)

    # 1) полосовой фильтр — полоса энергии QRS
    ny = fs/2
    b, a = butter(3, [band[0]/ny, min(band[1]/ny, 0.99)], btype='band')
    xf = filtfilt(b, a, x)

    # 2) производная + возведение в квадрат (подчёркивает крутые фронты QRS)
    d = np.diff(xf, append=xf[-1])
    sq = d**2

    # 3) интегрирование скользящим окном ~ длительность QRS
    w = max(3, int(round(integ_ms/1000*fs)))
    integ = np.convolve(sq, np.ones(w)/w, mode='same')

    # 4) ЛОКАЛЬНО адаптивный порог: доля от локального максимума
    #    (окно win_s секунд, шаг — половина окна)
    n = len(integ)
    W = max(int(win_s*fs), w*4)
    thr = np.zeros(n)
    for start in range(0, n, W//2):
        end = min(start+W, n)
        seg = integ[start:end]
        if len(seg) == 0:
            continue
        local = k*np.percentile(seg, 98)      # 98-й процентиль ≈ уровень QRS
        thr[start:end] = np.maximum(thr[start:end], local)
    thr[thr == 0] = k*np.percentile(integ, 98)

    # 5) поиск пиков с рефрактерным периодом
    dist = max(1, int(round(min_rr_ms/1000*fs)))
    cand, _ = find_peaks(integ, height=thr, distance=dist)

    # 6) уточнение положения по максимуму |полосового сигнала|
    half = max(2, int(round(refine_ms/1000*fs/2)))
    out = []
    for c in cand:
        a0, b0 = max(0, c-half), min(len(xf), c+half+1)
        out.append(a0 + int(np.argmax(np.abs(xf[a0:b0]))))
    return np.array(sorted(set(out)), dtype=int), xf


def suppress_qrs(sig, r_idx, fs, before_ms=50, after_ms=100):
    out = sig.copy()
    b = int(round(before_ms/1000*fs)); a = int(round(after_ms/1000*fs))
    for r in r_idx:
        i0, i1 = max(0, r-b), min(len(sig)-1, r+a)
        if i1-i0 < 2: continue
        out[i0:i1+1] = np.interp(np.arange(i0, i1+1), [i0, i1], [sig[i0], sig[i1]])
    return out


def halfwave(lam):
    """Вейвлет «полуволна»: положительный полупериод синуса
       с симметричными отрицательными крыльями (нулевое среднее)."""
    if lam % 2 == 0: lam += 1
    pos = np.sin(np.linspace(0, math.pi, lam))
    area = float(np.trapezoid(pos)) if hasattr(np, 'trapezoid') else float(np.trapz(pos))
    wing = max(1, lam//4)
    w = np.zeros(lam + 2*wing)
    w[:wing] = -area/(2*wing)
    w[wing:wing+lam] = pos
    w[wing+lam:] = -area/(2*wing)
    return w - w.mean()


def detect_pt(sig, r_idx, fs, wave_ms=85, thr_frac=0.3, local_s=30.0, noise_k=10.0,
              tol_ms=20.0, win=21, extra=True):
    """Возвращает индексы P и T (по одному на кардиоцикл, если найдены).
    Отклик — свёртка ЭКГ (с подавленными комплексами QRS) с вейвлетом «полуволна».
    T и основной поиск P: порог = thr_frac · 99-й процентиль положительного
    отклика по всей записи (без участков с аномально длинными RR).
    Дополнительный поиск P (для записей, где P в одной из фаз заметно меньше):
    порог по откликам в окнах поиска P в скользящем окне ±local_s секунд, но не
    ниже noise_k · уровня шума; такая находка принимается, только если расстояние
    P–R отличается от скользящей медианы по win циклам не больше tol_ms."""
    r_idx = np.asarray(r_idx, int)
    ecg = suppress_qrs(sig, r_idx, fs)
    lam = max(5, int(round(wave_ms/1000*fs)))
    w = halfwave(lam)
    conv = np.convolve(ecg, w[::-1], mode='same')
    n = len(conv)

    # порог по «здоровым» участкам (исключаем аномально длинные RR)
    rr = np.diff(r_idx)
    med = np.median(rr) if len(rr) else 0
    ok = np.ones(n, bool)
    for k in range(len(rr)):
        if med and rr[k] > 1.8*med:
            ok[r_idx[k]:r_idx[k+1]] = False
    pos = conv[(conv > 0) & ok]
    thr = thr_frac*np.percentile(pos, 99) if len(pos) else 0

    # окна поиска P и T
    wins = []
    for k in range(len(r_idx)):
        r = r_idx[k]
        r_prev = r_idx[k-1] if k > 0 else max(0, r-int(fs))
        r_next = r_idx[k+1] if k < len(r_idx)-1 else min(len(sig)-1, r+int(fs))
        # P: непосредственно перед QRS (P предшествует R на 100–300 мс)
        rr_p = r - r_prev
        lo = max(r_prev + rr_p//2, r - int(0.35*fs))
        ps, pe = lo, r - max(1, int(0.06*fs))
        # T: после QRS, до начала следующего цикла
        rr_n = r_next - r
        ts = r + max(int(0.12*fs), rr_n//8)
        te = min(r + int(0.6*rr_n), r_next - int(0.08*fs))
        wins.append((ps, pe, ts, te))

    P, T = [], []
    for ps, pe, ts, te in wins:
        if ps < pe <= len(conv):
            seg = conv[ps:pe]
            P.append(ps + int(np.argmax(seg)) if len(seg) and seg.max() > thr else -1)
        else:
            P.append(-1)
        if ts < te <= len(conv):
            seg = conv[ts:te]
            T.append(ts + int(np.argmax(seg)) if len(seg) and seg.max() > thr else -1)
        else:
            T.append(-1)
    P = np.array(P); T = np.array(T)

    if not extra:
        return P, T, ecg, conv
    # дополнительный поиск P: порог по окнам P в скользящем окне со страховкой от шума
    pmask = np.zeros(n, bool)
    for ps, pe, _, _ in wins:
        if ps < pe <= n:
            pmask[ps:pe] = True
    d = np.diff(ecg)
    noise = 1.4826*np.median(np.abs(d - np.median(d)))/np.sqrt(2)*np.sqrt(np.sum(w**2))
    L = int(local_s*fs)
    P2 = np.full(len(r_idx), -1)
    for k, (ps, pe, _, _) in enumerate(wins):
        if not (ps < pe <= n):
            continue
        i = r_idx[k]; a, b = max(0, i-L), min(n, i+L)
        sg = conv[a:b]; mk = (sg > 0) & ok[a:b] & pmask[a:b]
        thr_p = thr_frac*np.percentile(sg[mk], 99) if mk.sum() > 10 else thr
        seg = conv[ps:pe]
        if len(seg) and seg.max() > max(thr_p, noise_k*noise):
            P2[k] = ps + int(np.argmax(seg))
    off = np.where(P2 >= 0, (r_idx - P2)/fs*1000, np.nan)
    med_off = np.full(len(off), np.nan); h = win//2
    for i in range(len(off)):
        s = off[max(0, i-h):i+h+1]; s = s[np.isfinite(s)]
        if len(s) >= 3:
            med_off[i] = np.median(s)
    add = (P < 0) & (P2 >= 0) & (np.abs(off - med_off) <= tol_ms)
    P[add] = P2[add]
    return P, T, ecg, conv


def phase_bounds(marks):
    """Границы подъёма стола по протоколу пробы. Метки «Этап»: около 05:00 —
    начало подъёма, около 05:10–05:20 — конец подъёма, около 10:00 — конец
    записи. Если в зоне подъёма несколько меток, первая — начало, последняя —
    конец; если одна и она позже 05:10 — это конец подъёма, начало принимается
    на 15 с раньше. Возвращает (начало подъёма, конец подъёма), с."""
    mid = sorted(t for t in marks if 250 < t < 340)
    if len(mid) >= 2:
        return mid[0], mid[-1]
    if len(mid) == 1:
        m = mid[0]
        return (m - 15.0, m) if m >= 310 else (m, m + 15.0)
    return 300.0, 315.0


def band_ecg(sig,fs,lo=0.5,hi=40.0):
    b,a=butter(2,[lo/(fs/2),hi/(fs/2)],'band'); return filtfilt(b,a,sig)


def parabolic(y,i):
    """Вершина параболы через три точки вокруг i: поправка в долях отсчёта."""
    if i<=0 or i>=len(y)-1: return float(i)
    a,b,c=y[i-1],y[i],y[i+1]; den=a-2*b+c
    return float(i) if den==0 else i+0.5*(a-c)/den


def refine_r(y,r,fs,half_ms=8):
    """Уточнение вершины R: максимум |ЭКГ| рядом с отметкой + парабола."""
    h=int(half_ms/1000*fs); out=np.full(len(r),np.nan)
    for k,ri in enumerate(r):
        a,b=max(1,ri-h),min(len(y)-1,ri+h+1); seg=np.abs(y[a:b]); i=int(np.argmax(seg))
        out[k]=a+parabolic(seg,i)
    return out


def refine_p(y,P,fs,half_ms=60,search_ms=20,iters=2):
    """Уточнение положения P по шаблону: шаблон — среднее участков ЭКГ вокруг
    найденных P; в каждом цикле — сдвиг максимальной корреляции с шаблоном
    (в пределах ±search_ms) с параболической интерполяцией."""
    P=np.asarray(P); h=int(half_ms/1000*fs); s=int(search_ms/1000*fs); idx=np.where(P>=0)[0]
    pos=P.astype(float).copy(); pos[P<0]=np.nan
    for _ in range(iters):
        segs=[]
        for k in idx:
            c=int(round(pos[k]))
            if c-h-s>=0 and c+h+s<len(y): segs.append(y[c-h:c+h+1]-np.mean(y[c-h:c+h+1]))
        if len(segs)<20: return pos
        tpl=np.mean(segs,0); tpl=tpl-tpl.mean(); tn=np.linalg.norm(tpl)
        new=pos.copy()
        for k in idx:
            c=int(round(pos[k]))
            if not (c-h-s>=0 and c+h+s<len(y)): continue
            cc=np.empty(2*s+1)
            for j,l in enumerate(range(-s,s+1)):
                w=y[c+l-h:c+l+h+1]; w=w-w.mean(); nw=np.linalg.norm(w)
                cc[j]=np.dot(w,tpl)/(nw*tn) if nw>0 else -1
            i=int(np.argmax(cc)); new[k]=c+(parabolic(cc,i)-s)
        pos=new
    return pos


def build_series_ch(path, ch, refine=True, extra_p=True):
    """Ряды RR, PP и PR одного отведения. Положения R и P уточняются точнее
    одного отсчёта: R — параболической интерполяцией вершины, P — по шаблону
    зубца P записи (корреляция в пределах ±20 мс с интерполяцией). Все три ряда
    строятся из одних и тех же уточнённых точек, поэтому тождество
    RR(k) = PP(k) + PR(k+1) − PR(k) выполняется точно."""
    f = pyedflib.EdfReader(path)
    ann = f.readAnnotations()
    f.close()
    marks = [float(t) for t in ann[0]]
    p1e, p2b = phase_bounds(marks)

    sig, fs = read_ecg(path, ch=ch)
    r, _ = detect_r(sig, fs)
    P, T, _, _ = detect_pt(sig, r, fs, extra=extra_p)
    if refine:
        y = band_ecg(sig, fs)
        r_ref = refine_r(y, r, fs)
        p_ref = refine_p(y, P, fs)
    else:                                   # вариант без уточнения: точки — индексы отсчётов
        r_ref = r.astype(float)
        p_ref = np.where(P >= 0, P, np.nan).astype(float)

    t_r = r_ref/fs
    rr = np.diff(r_ref)/fs*1000
    has_p = (P > 0) & np.isfinite(p_ref)
    pr = np.full(len(r), np.nan)
    pr[has_p] = (r_ref[has_p] - p_ref[has_p])/fs*1000
    pp = np.full(len(r), np.nan)
    idx = np.where(has_p)[0]
    for a, b in zip(idx[:-1], idx[1:]):
        if b == a + 1:
            pp[a] = (p_ref[b] - p_ref[a])/fs*1000

    return dict(t_r=t_r, rr=rr, t_rr=t_r[:-1], pr=pr, pp=pp, r_ref=r_ref, p_ref=p_ref, fs=fs,
                p1e=p1e, p2b=p2b, n_r=len(r),
                n_pr=int(np.isfinite(pr).sum()), n_pp=int(np.isfinite(pp).sum()))


def phase_windows(v):
    return {'покой': (30.0, v['p1e'] - 5.0), 'наклон': (v['p2b'] + 15.0, v['p2b'] + 275.0)}


def clean_rr(rr):
    x = np.asarray(rr, float).copy(); base = np.where(np.isfinite(x), x, np.nanmedian(x))
    x[np.abs(x - median_filter(base, 11, mode='nearest')) > ARTIFACT_MS] = np.nan
    return x


def clean_outliers(x):
    x = np.asarray(x, float).copy(); ok = np.isfinite(x)
    if ok.sum() < 30:
        return x
    xs = x[ok]; f = xs - np.convolve(np.pad(xs, 7, mode='edge'), np.ones(15) / 15, 'valid')
    med = np.median(f); mad = 1.4826 * np.median(np.abs(f - med)); idx = np.where(ok)[0]
    x[idx[np.abs(f - med) > OUTLIER_MAD * mad]] = np.nan
    return x


def match_beats(ra, rb, fs, tol_ms=40):
    j = np.searchsorted(rb, ra); j = np.clip(j, 1, len(rb) - 1)
    j = np.where(np.abs(rb[j - 1] - ra) < np.abs(rb[j] - ra), j - 1, j)
    return j, np.abs(rb[j] - ra) <= tol_ms / 1000 * fs


def crosslead_noise(v2, v3):
    """Погрешность PR на одно отведение: робастное СКО поцикловой разницы PR
    в отведениях II и III, делённое на √2 (оценка сверху)."""
    j, ok = match_beats(v2['r_ref'], v3['r_ref'], v2['fs'])
    d = v2['pr'][ok] - v3['pr'][j[ok]]; d = d[np.isfinite(d)]; d = d - np.median(d)
    mad = 1.4826 * np.median(np.abs(d)); d = d[np.abs(d) <= 5 * max(mad, 1.0)]
    return float(np.std(d) / np.sqrt(2))


def decomposition_noise(v):
    """Разложение дисперсии разностей соседних PR: для белого шума автокорреляция
    разностей на первом шаге равна −0,5; доля шума = −r1 / 0,5. Если r1 ≥ 0,
    медленные изменения PR перекрывают вклад шума, и оценка этим способом не
    определена (NaN); такие записи в медианы не входят."""
    pr = np.asarray(v['pr'], float); d = pr[1:] - pr[:-1]; ok = np.isfinite(d[:-1]) & np.isfinite(d[1:])
    a, b = d[:-1][ok], d[1:][ok]; q = np.percentile(np.abs(np.r_[a, b]), 99); keep = (np.abs(a) <= q) & (np.abs(b) <= q)
    a, b = a[keep], b[keep]; r1 = float(np.corrcoef(a, b)[0, 1]); sd = float(np.std(np.r_[a, b], ddof=1))
    frac = min(1.0, -r1 / 0.5) if r1 < 0 else np.nan
    return dict(r1=r1, sd_diff=sd, frac=frac, noise_sd=sd * np.sqrt(frac) / np.sqrt(2), own_sd=sd * np.sqrt(1 - frac))


# ═════════════ ЧАСТЬ 2. Вспомогательные функции пункта 0 ═════════════

def movavg(x, k=SMOOTH):
    """Центрированное скользящее среднее по k циклам с пропусками: значение
    определено, если в окне не меньше половины циклов."""
    x = np.asarray(x, float); ok = np.isfinite(x)
    s = np.convolve(np.where(ok, x, 0.0), np.ones(k), 'same'); c = np.convolve(ok.astype(float), np.ones(k), 'same')
    out = np.full_like(x, np.nan); m = c >= k // 2 + 1; out[m] = s[m] / c[m]
    return out


def detrend_lin(t, x):
    """Удаление линейного тренда по времени (по определённым точкам)."""
    x = np.asarray(x, float).copy(); ok = np.isfinite(x)
    if ok.sum() < 10:
        return x
    a, b = np.polyfit(t[ok], x[ok], 1); x[ok] = x[ok] - (a * t[ok] + b)
    return x


def detrend_x(t, x, mode):
    """Удаление тренда внутри фазы: True/'linear' — линейный; 'ma101' — медленный
    тренд адаптации (скользящее среднее по 101 циклу); False — только среднее."""
    if mode is True or mode == 'linear':
        return detrend_lin(t, x)
    if mode == 'ma101':
        return np.asarray(x, float) - movavg(x, 101)
    return np.asarray(x, float) - np.nanmean(x)


def corr(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 30 or np.std(a[ok]) == 0 or np.std(b[ok]) == 0:
        return np.nan
    return float(np.corrcoef(a[ok], b[ok])[0, 1])


def shift_surrogate_p(a, b, rng, n=N_SURR, min_shift=MIN_SHIFT):
    """Значимость корреляции двух автокоррелированных рядов: доля циклических
    сдвигов одного ряда относительно другого, дающих |r| не меньше наблюдаемой."""
    r0 = corr(a, b); L = len(a)
    if not np.isfinite(r0) or L < 3 * min_shift:
        return np.nan
    sh = rng.integers(min_shift, L - min_shift, n)
    rs = np.array([corr(a, np.roll(b, s)) for s in sh]); rs = rs[np.isfinite(rs)]
    return float((np.sum(np.abs(rs) >= abs(r0)) + 1) / (len(rs) + 1))


def noise_in_mask(v2, v3, mask):
    """Погрешность PR и собственная изменчивость внутри маски циклов — по тем же
    правилам, что в п. 4. (1) Разложение дисперсии разностей соседних PR: доля
    шума = −r1 / 0,5; при r1 ≥ 0 оценка не определена (NaN). (2) Разница PR одного
    цикла в отведениях II и III, делённая на √2 (оценка сверху); собственная
    изменчивость разностей по ней — √(СКО² разностей − 2·погрешность²) (оценка
    снизу). (3) Разброс положения R между отведениями (по разностям соседних
    циклов), мс."""
    pr = np.asarray(v2['pr'], float); d = pr[1:] - pr[:-1]; d[~(mask[1:] & mask[:-1])] = np.nan
    ok = np.isfinite(d[:-1]) & np.isfinite(d[1:]); out = dict(noise_dec=np.nan, own_dec=np.nan, sd_diff=np.nan, noise_cl=np.nan, own_cl=np.nan, r_jitter=np.nan)
    if ok.sum() < 50:
        return out
    a, b = d[:-1][ok], d[1:][ok]; q = np.percentile(np.abs(np.r_[a, b]), 99); keep = (np.abs(a) <= q) & (np.abs(b) <= q)
    a, b = a[keep], b[keep]; r1 = float(np.corrcoef(a, b)[0, 1]); sd = float(np.std(np.r_[a, b], ddof=1)); out['sd_diff'] = sd
    if r1 < 0:
        frac = min(1.0, -r1 / 0.5); out['noise_dec'] = sd * np.sqrt(frac) / np.sqrt(2); out['own_dec'] = sd * np.sqrt(1 - frac)
    if v3 is not None:
        j, okb = match_beats(v2['r_ref'], v3['r_ref'], v2['fs']); okb = okb & mask
        dd = v2['pr'][okb] - v3['pr'][j[okb]]; dd = dd[np.isfinite(dd)]
        if len(dd) >= 50:
            dd = dd - np.median(dd); mad = 1.4826 * np.median(np.abs(dd)); dd = dd[np.abs(dd) <= 5 * max(mad, 1.0)]
            out['noise_cl'] = float(np.std(dd) / np.sqrt(2)); out['own_cl'] = float(np.sqrt(max(sd ** 2 - 2 * out['noise_cl'] ** 2, 0.0)))
            dr = (v2['r_ref'][okb] - v3['r_ref'][j[okb]]) / v2['fs'] * 1000; ddr = np.diff(dr); md = np.median(ddr)
            ddr = ddr[np.abs(ddr - md) <= 5 * max(1.4826 * np.median(np.abs(ddr - md)), 0.1)]; out['r_jitter'] = float(np.std(ddr) / 2)
    return out


def p_template(path, ch, p_ref, t_r, win):
    """Средняя форма зубца P в окне фазы (ЭКГ 0,5–40 Гц, ±60 мс вокруг уточнённой
    точки P): размах шаблона, сам шаблон и положение относительно уточнённой
    точки P двух других точек отсчёта — вершины шаблона и центра его верхней
    половины, мс. Если положение различается в покое и при наклоне, изменение
    среднего PR при смене позы зависит от выбора точки отсчёта на зубце P."""
    sig, fs = read_ecg(path, ch); y = band_ecg(sig, fs); h = int(0.06 * fs); segs = []
    for k in np.where(np.isfinite(p_ref) & (t_r >= win[0]) & (t_r <= win[1]))[0]:
        c = int(round(p_ref[k]))
        if c - h >= 0 and c + h < len(y):
            s = y[c - h:c + h + 1]; segs.append(s - np.mean(s[:int(0.015 * fs)]))
    if len(segs) < 20:
        return np.nan, None, (np.nan, np.nan)
    tpl = np.mean(segs, 0); i = int(np.argmax(tpl))
    top = np.clip(tpl - 0.5 * tpl.max(), 0, None)          # верхняя половина зубца
    cen = float(np.sum(top * np.arange(len(tpl))) / np.sum(top))
    return float(tpl.max() - tpl.min()), tpl, (float((parabolic(tpl, i) - h) / fs * 1000), float((cen - h) / fs * 1000))


def wilcox(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return float(stats.wilcoxon(x).pvalue) if len(x) >= 6 and np.any(x != 0) else np.nan


def med(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan


# ═════════════ ЧАСТЬ 3. Расчёт по записи ═════════════

def prepare_record(v):
    """Очищенные ряды записи и маски фаз. RR_prev(k) — цикл, предшествующий
    зубцу P k-го цикла; PP(k) − RR(k) = PR(k) − PR(k+1)."""
    t = v['t_r']; n = len(t)
    rr = clean_rr(v['rr']); pr = clean_outliers(v['pr']); pp = clean_outliers(v['pp'])
    rr_prev = np.r_[np.nan, rr]; rr_next = np.r_[rr, np.nan]
    W = phase_windows(v); masks = {}; cover = {}
    for ph, (a, b) in W.items():
        m = (t >= a) & (t <= b); masks[ph] = m
        cover[ph] = float(np.isfinite(pr[m]).mean()) if m.sum() else 0.0
    return dict(t=t, n=n, rr=rr_next, rr_prev=rr_prev, pr=pr, pp=pp, masks=masks, cover=cover, W=W)


def phase_stats(R, ph, rng, smooth=SMOOTH, detrend=True, surr=True):
    """Все показатели пункта 0 внутри одной фазы записи."""
    m = R['masks'][ph]; t = R['t'][m]
    pr = R['pr'][m]; rrp = R['rr_prev'][m]; rr = R['rr'][m]; pp = R['pp'][m]
    out = {}
    # 1. расхождение PP и RR
    disc = pp - rr
    out['sd_rr'] = float(np.nanstd(rr)); out['sd_disc'] = float(np.nanstd(disc)); out['r_pp_rr'] = corr(pp, rr)
    # 3. медленная и быстрая составляющие
    prd, rrd = detrend_x(t, pr, detrend), detrend_x(t, rrp, detrend)
    slow_pr, slow_rr = movavg(prd, smooth), movavg(rrd, smooth)
    fast_pr, fast_rr = prd - slow_pr, rrd - slow_rr
    fast_rr_next = detrend_x(t, rr, detrend); fast_rr_next = fast_rr_next - movavg(fast_rr_next, smooth)
    out['sd_pr'] = float(np.nanstd(prd)); out['sd_dpr'] = float(np.nanstd(np.diff(prd)))
    out['ratio_diff'] = out['sd_dpr'] / out['sd_pr'] if out['sd_pr'] > 0 else np.nan
    out['sd_slow'] = float(np.nanstd(slow_pr)); out['sd_fast'] = float(np.nanstd(fast_pr))
    # 4. связь с ритмом
    out['r_raw'] = corr(prd, rrd); out['r_slow'] = corr(slow_pr, slow_rr)
    out['r_fast_prev'] = corr(fast_pr, fast_rr); out['r_fast_next'] = corr(fast_pr, fast_rr_next)
    out['p_slow_surr'] = shift_surrogate_p(slow_pr, slow_rr, rng) if surr else np.nan
    out['var_slow'] = float(np.nanvar(slow_pr)); out['smooth'] = smooth
    out['mean_pr'] = float(np.nanmean(pr)); out['mean_rr'] = float(np.nanmean(rr))
    # 6. блоки (по тренд-удалённому ряду)
    for L in BLOCKS:
        means = []
        for s in range(0, len(prd) - L + 1, L):
            seg = prd[s:s + L]
            if np.isfinite(seg).sum() >= 0.8 * L:
                means.append(np.nanmean(seg))
        out[f'block{L}_sd'] = float(np.std(means, ddof=1)) if len(means) >= 3 else np.nan
        out[f'block{L}_fast_expect'] = out['sd_fast'] / np.sqrt(L)
    return out


def record_stats(name, v2, v3, rng, smooth=SMOOTH, detrend=True, with_noise=True, surr=True):
    R = prepare_record(v2); res = {'rec': name, 'cover': R['cover']}
    whole = np.ones(R['n'], bool)
    # вся запись (как в прежней версии): исходные и сглаженные ряды без разбиения на фазы
    res['whole_r_raw'] = corr(R['pr'], R['rr_prev']); res['whole_r_slow'] = corr(movavg(R['pr'], smooth), movavg(R['rr_prev'], smooth))
    # вся запись после вычитания средних каждой фазы (без переходного участка)
    pr_c = np.full(R['n'], np.nan); rr_c = np.full(R['n'], np.nan)
    for ph, m in R['masks'].items():
        pr_c[m] = R['pr'][m] - np.nanmean(R['pr'][m]); rr_c[m] = R['rr_prev'][m] - np.nanmean(R['rr_prev'][m])
    res['whole_r_slow_centered'] = corr(movavg(pr_c, smooth), movavg(rr_c, smooth))
    res['phase'] = {}
    for ph in ('покой', 'наклон'):
        if R['cover'][ph] < COVERAGE:
            res['phase'][ph] = None; continue
        st = phase_stats(R, ph, rng, smooth, detrend, surr)
        if with_noise:
            st.update(noise_in_mask(v2, v3, R['masks'][ph]))
            sh = (st['noise_cl'] ** 2 / smooth) / st['var_slow'] if np.isfinite(st['noise_cl']) and st['var_slow'] > 0 else np.nan
            st['slow_noise_share'] = sh
            # доля погрешности в дисперсии разностей соседних PR (то есть расхождения PP и RR):
            # по разнице II–III — 2·погрешность² / СКО² разностей; по разложению — −2·r1
            st['fast_noise_share'] = float(min(1.0, st['noise_cl'] ** 2 * (1 - 1 / smooth) / st['sd_fast'] ** 2)) if np.isfinite(st['noise_cl']) and st['sd_fast'] > 0 else np.nan
            st['diff_noise_share_cl'] = float(min(1.0, 2 * st['noise_cl'] ** 2 / st['sd_diff'] ** 2)) if np.isfinite(st['noise_cl']) and st['sd_diff'] > 0 else np.nan
            st['diff_noise_share_dec'] = float(2 * st['noise_dec'] ** 2 / (2 * st['noise_dec'] ** 2 + st['own_dec'] ** 2)) if np.isfinite(st['noise_dec']) else np.nan
            st['r_slow_corr'] = float(np.clip(st['r_slow'] / np.sqrt(1 - min(sh, 0.9)), -1, 1)) if np.isfinite(sh) else np.nan
        res['phase'][ph] = st
    if with_noise:
        nw = noise_in_mask(v2, v3, whole); res['noise_dec_whole'] = nw['noise_dec']; res['own_dec_whole'] = nw['own_dec']
        res['noise_cl_whole'] = nw['noise_cl']
    return res


# ═════════════ ЧАСТЬ 4. Синтетическая проверка метода ═════════════

def synthetic_check(S, noise_sd_rec, reps=20, seed=SEED):
    """На настоящих рядах RR каждой записи строится искусственный PR с известным
    ответом: (а) медленная составляющая PR не связана с RR, но средний PR при
    смене позы сдвигается так же, как в настоящей записи; (б) медленная
    составляющая PR внутри фаз частично следует за RR на всех медленных масштабах;
    (в) PR следует за RR только через общий медленный дрейф (масштаб больше
    примерно 100 циклов), а на масштабах 15–100 циклов с RR не связан. Добавляется
    шум измерения с погрешностью этой записи. Каждый ряд обрабатывается основным
    расчётом (удаление линейного тренда) и с удалением медленного тренда
    (среднее по 101 циклу)."""
    rng = np.random.default_rng(seed); out = {'а': [], 'б': [], 'в': []}
    for name, v in S.items():
        R = prepare_record(v); n = R['n']; rrp = R['rr_prev']
        if any(R['cover'][ph] < COVERAGE for ph in R['masks']):
            continue
        noise = noise_sd_rec[name]
        shift = np.nanmean(R['pr'][R['masks']['наклон']]) - np.nanmean(R['pr'][R['masks']['покой']])
        step = np.where(R['t'] > (R['W']['покой'][1] + R['W']['наклон'][0]) / 2, shift, 0.0)
        sd_slow = np.nanstd(movavg(R['pr'], SMOOTH)[R['masks']['покой']])
        for sc in ('а', 'б', 'в'):
            for _ in range(reps):
                e = rng.standard_normal(n); ar = np.zeros(n)
                for k in range(1, n):
                    ar[k] = 0.95 * ar[k - 1] + e[k]
                ar = ar / np.std(ar) * sd_slow
                if sc in ('б', 'в'):
                    cpl = np.zeros(n)
                    for ph, m in R['masks'].items():
                        x = movavg(detrend_lin(R['t'][m], rrp[m]), SMOOTH if sc == 'б' else 101)
                        cpl[m] = np.nan_to_num(x / np.nanstd(x)) * sd_slow
                    ar = 0.8 * cpl + 0.6 * ar
                pr = 150 + step + ar + rng.standard_normal(n) * noise
                pr[~np.isfinite(R['pr'])] = np.nan
                vv = dict(v); vv['pr'] = pr
                rs = record_stats(name, vv, None, rng, with_noise=False)
                rm = record_stats(name, vv, None, rng, with_noise=False, detrend='ma101', surr=False)
                out[sc].append(dict(rec=name, whole=rs['whole_r_slow'], centered=rs['whole_r_slow_centered'],
                                    rest=rs['phase']['покой']['r_slow'], tilt=rs['phase']['наклон']['r_slow'],
                                    rest101=rm['phase']['покой']['r_slow'], tilt101=rm['phase']['наклон']['r_slow'],
                                    p_rest=rs['phase']['покой']['p_slow_surr'], p_tilt=rs['phase']['наклон']['p_slow_surr']))
    summ = {}
    for sc, L in out.items():
        a = lambda k: np.array([d[k] for d in L], float)
        summ[sc] = dict(n=len(L), whole=med(a('whole')), centered=med(a('centered')), rest=med(a('rest')), tilt=med(a('tilt')),
                        rest101=med(a('rest101')), tilt101=med(a('tilt101')),
                        detect_rest=float(np.nanmean(a('p_rest') < 0.05)), detect_tilt=float(np.nanmean(a('p_tilt') < 0.05)))
    return summ


# ═════════════ ЧАСТЬ 5. Сводка ═════════════

def collect(stats_list, key, ph):
    return np.array([s['phase'][ph].get(key, np.nan) if s['phase'].get(ph) else np.nan for s in stats_list], float)


def paired(stats_list, key):
    a, b = collect(stats_list, key, 'покой'), collect(stats_list, key, 'наклон'); ok = np.isfinite(a) & np.isfinite(b)
    d = b[ok] - a[ok]
    return dict(n=int(ok.sum()), rest=med(a[ok]), tilt=med(b[ok]), up=int(np.sum(d > 0)), p=wilcox(d))


def summarize(stats_list):
    S = {}
    for ph in ('покой', 'наклон'):
        g = lambda k: collect(stats_list, k, ph)
        S[ph] = {k: med(g(k)) for k in ('sd_rr', 'sd_disc', 'r_pp_rr', 'noise_dec', 'noise_cl', 'own_dec', 'own_cl', 'r_jitter', 'sd_pr', 'sd_dpr',
                                         'ratio_diff', 'sd_slow', 'sd_fast', 'r_raw', 'r_slow', 'r_slow_corr', 'slow_noise_share',
                                         'r_fast_prev', 'r_fast_next', 'mean_pr', 'fast_noise_share')}
        S[ph]['n'] = int(np.isfinite(g('r_slow')).sum())
        S[ph]['n_dec'] = int(np.isfinite(g('noise_dec')).sum())
        S[ph]['own_gt_noise_dec'] = int(np.nansum(g('own_dec') > g('noise_dec')))
        S[ph]['own_gt_noise_cl'] = int(np.nansum(g('own_cl') > g('noise_cl'))); S[ph]['n_cl'] = int(np.isfinite(g('noise_cl')).sum())
        S[ph]['slow_vs_noise'] = med(g('sd_slow') / (g('noise_cl') / np.sqrt(SMOOTH)))
        for k in ('r_raw', 'r_slow', 'r_fast_prev', 'r_fast_next'):
            x = g(k); S[ph][k + '_pos'] = int(np.nansum(x > 0)); S[ph][k + '_p'] = wilcox(x)
        S[ph]['surr_sig'] = int(np.nansum(g('p_slow_surr') < 0.05)); S[ph]['ratio_lt'] = int(np.nansum(g('ratio_diff') < np.sqrt(2)))
        for L in BLOCKS:
            S[ph][f'block{L}'] = med(g(f'block{L}_sd')); S[ph][f'block{L}_fast'] = med(g(f'block{L}_fast_expect'))
            S[ph][f'block{L}_noise'] = med(g('noise_cl') / np.sqrt(L))
            S[ph][f'block{L}_exceed'] = int(np.nansum(g(f'block{L}_sd') > g(f'block{L}_fast_expect')))
    w = lambda k: np.array([s[k] for s in stats_list], float)
    S['вся запись'] = dict(r_raw=med(w('whole_r_raw')), r_slow=med(w('whole_r_slow')), r_slow_centered=med(w('whole_r_slow_centered')),
                           r_slow_pos=int(np.nansum(w('whole_r_slow') > 0)), r_slow_p=wilcox(w('whole_r_slow')),
                           r_cent_p=wilcox(w('whole_r_slow_centered')), n=int(np.isfinite(w('whole_r_slow')).sum()))
    if 'noise_dec_whole' in stats_list[0]:
        S['вся запись'].update(noise_dec=med(w('noise_dec_whole')), own_dec=med(w('own_dec_whole')), noise_cl=med(w('noise_cl_whole')))
    S['изменение'] = {k: paired(stats_list, k) for k in ('r_slow', 'r_slow_corr', 'r_raw', 'r_fast_prev', 'r_fast_next', 'sd_slow', 'own_dec', 'own_cl',
                                                     'noise_dec', 'noise_cl', 'mean_pr', 'sd_disc', 'diff_noise_share_cl', 'diff_noise_share_dec')}
    # доля погрешности меньше половины — на тех же записях, что и сравнение фаз (обе фазы определены)
    for k in ('diff_noise_share_cl', 'diff_noise_share_dec'):
        a, b = collect(stats_list, k, 'покой'), collect(stats_list, k, 'наклон'); ok = np.isfinite(a) & np.isfinite(b)
        S['изменение'][k].update(rest_lt05=int(np.sum(a[ok] < 0.5)), tilt_lt05=int(np.sum(b[ok] < 0.5)))
    return S


# ═════════════ ЧАСТЬ 6. Рисунки ═════════════

def fig_example(v, name, path):
    R = prepare_record(v); t = R['t']
    fig, ax = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True)
    ax[0].plot(t, R['rr_prev'], color='0.7', lw=0.6); ax[0].plot(t, movavg(R['rr_prev']), color='tab:blue', lw=1.6)
    ax[1].plot(t, R['pr'], color='0.7', lw=0.6); ax[1].plot(t, movavg(R['pr']), color='tab:red', lw=1.6)
    for a in ax:
        for ph, (lo, hi) in R['W'].items():
            a.axvspan(lo, hi, color='tab:green' if ph == 'покой' else 'tab:orange', alpha=0.08)
    ax[0].set_ylabel('RR, мс'); ax[1].set_ylabel('PR, мс'); ax[1].set_xlabel('время, с')
    ax[0].set_title(f'Запись {name}: ряды и их медленные составляющие (скользящее среднее по {SMOOTH} циклам); '
                    'зелёный фон — покой, оранжевый — наклон', fontsize=9)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def fig_rest_tilt(stats_list, stats_101, path):
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
    for ax, SL, ttl in ((axs[0], stats_list, 'а — основной расчёт (удалён линейный тренд)'),
                        (axs[1], stats_101, 'б — удалён медленный тренд (среднее по 101 циклу)')):
        a, b = collect(SL, 'r_slow', 'покой'), collect(SL, 'r_slow', 'наклон')
        for i in range(len(a)):
            if np.isfinite(a[i]) and np.isfinite(b[i]):
                ax.plot([0, 1], [a[i], b[i]], color='0.6', lw=0.8, marker='o', ms=4)
        ax.plot([0, 1], [med(a), med(b)], color='k', lw=2.2, marker='s', ms=6)   # медианы по всем записям с данной фазой
        ax.axhline(0, color='k', lw=0.6); ax.set_xticks([0, 1]); ax.set_xticklabels(['покой', 'наклон'])
        ax.set_xlim(-0.3, 1.3); ax.set_title(ttl, fontsize=9)
    w = np.array([s['whole_r_slow'] for s in stats_list])
    axs[0].scatter(np.full(len(w), 1.2), w, color='tab:purple', s=14)          # вся запись без разбиения на фазы
    axs[0].set_xlim(-0.3, 1.35)
    axs[0].set_ylabel('r медленных составляющих PR и RR')
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def fig_blocks(S, path):
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
    for a, ph in zip(ax, ('покой', 'наклон')):
        L = np.array(BLOCKS)
        a.plot(L, [S[ph][f'block{l}'] for l in L], 'o-', color='tab:red', label='наблюдаемый разброс средних блоков')
        a.plot(L, [S[ph][f'block{l}_fast'] for l in L], 's--', color='0.4', label='ожидаемый без медленных изменений')
        a.plot(L, [S[ph][f'block{l}_noise'] for l in L], '^:', color='0.6', label='ожидаемый от погрешности измерения')
        a.set_xscale('log'); a.set_xticks(L); a.set_xticklabels([str(l) for l in L]); a.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        a.set_title(ph); a.set_xlabel('длина блока, циклов')
    ax[0].set_ylabel('СКО среднего PR по блокам, мс'); ax[0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer): return int(o)
    return o


# ═════════════ ЧАСТЬ 7. Запуск ═════════════

def run_p0(data_dir, out_dir='results_p0'):
    os.makedirs(out_dir, exist_ok=True)
    paths = sorted(glob.glob(os.path.join(data_dir, '00*_ecg.edf'))); names = [os.path.basename(p)[:4] for p in paths]
    if not paths:
        raise SystemExit(f'В папке {data_dir} нет файлов 00*_ecg.edf')
    print(f'Записей: {len(paths)}. Строю ряды (отведения II и III, лучшее отведение, вариант без уточнения)...', flush=True)
    B = lambda ch_fn, **kw: {n: build_series_ch(p, ch_fn(n), **kw) for n, p in zip(names, paths)}
    S2 = B(lambda n: LEAD_II_CHANNEL); S3 = B(lambda n: LEAD_III_CHANNEL)
    rng = np.random.default_rng(SEED)
    main = [record_stats(n, S2[n], S3[n], rng) for n in names]
    summ = {'основной': summarize(main)}
    M = summ['основной']; W_ = M['вся запись']; C = M['изменение']
    print(f"\nОхват PR по фазам (доля циклов): ниже {COVERAGE:.0%} — " +
          (', '.join(f"{s['rec']} {ph} ({s['cover'][ph]:.0%}; зубец P найден в {np.isfinite(S2[s['rec']]['pr'][prepare_record(S2[s['rec']])['masks'][ph]]).mean():.0%} циклов, "
                    f"разница — исключённые выбросы)" for s in main for ph in ('покой', 'наклон') if s['cover'][ph] < COVERAGE) or 'нет'))
    print("\n1. Расхождение PP и RR (медианы)")
    for ph in ('покой', 'наклон'):
        m = M[ph]; print(f"  {ph}: СКО RR {m['sd_rr']:.1f} мс, СКО (PP − RR) {m['sd_disc']:.2f} мс ({m['sd_disc']/m['sd_rr']:.1%} от СКО RR), r(PP, RR) {m['r_pp_rr']:.4f}")
    c = C['sd_disc']; print(f"  СКО (PP − RR) при наклоне (записи с обеими фазами): {c['rest']:.2f} → {c['tilt']:.2f} мс, рост у {c['up']}/{c['n']}, p = {c['p']:.4f}")
    print("\n2. Погрешность и собственная изменчивость PR (медианы, мс)")
    print(f"  вся запись: погрешность — разложение {W_['noise_dec']:.2f}, разница II–III {W_['noise_cl']:.2f}; собственная {W_['own_dec']:.2f}")
    for k, lab in (('noise_dec', 'погрешность, разложение'), ('noise_cl', 'погрешность, разница II–III'),
                   ('own_dec', 'собственная изменчивость разностей, разложение'), ('own_cl', 'собственная изменчивость разностей, разница II–III')):
        c = C[k]; print(f"  {lab}: покой {c['rest']:.2f}, наклон {c['tilt']:.2f} (записей с обеими фазами {c['n']}); при наклоне больше у {c['up']}/{c['n']}, p = {c['p']:.4f}")
    for ph in ('покой', 'наклон'):
        m = M[ph]; print(f"  {ph}, все записи с этой фазой: погрешность — разложение {m['noise_dec']:.2f} (определена у {m['n_dec']}/{m['n']}), "
                         f"разница II–III {m['noise_cl']:.2f} (у {m['n_cl']}/{m['n']}); разброс R между отведениями {m['r_jitter']:.2f}")
    print("  Доля погрешности в дисперсии разностей соседних PR (= дисперсии расхождения PP и RR), медианы по записям с обеими фазами:")
    for k, lab in (('diff_noise_share_cl', 'по разнице II–III'), ('diff_noise_share_dec', 'по разложению')):
        c = C[k]; print(f"    {lab}: покой {c['rest']:.2f} (меньше половины у {c['rest_lt05']}/{c['n']}), наклон {c['tilt']:.2f} "
                        f"(меньше половины у {c['tilt_lt05']}/{c['n']}); при наклоне больше у {c['up']}/{c['n']}, p = {c['p']:.4f}")
    print("\n3. Медленная и быстрая составляющие внутри фаз (медианы)")
    for ph in ('покой', 'наклон'):
        m = M[ph]; print(f"  {ph}: СКО(ΔPR)/СКО(PR) {m['ratio_diff']:.2f} (белый шум 1,41; ниже у {m['ratio_lt']}/{m['n']}); "
                         f"СКО медленной {m['sd_slow']:.2f} мс = {m['slow_vs_noise']:.1f}× шум после сглаживания (погрешность по разнице II–III), "
                         f"доля шума в дисперсии медленной части {m['slow_noise_share']:.2f}; СКО быстрой {m['sd_fast']:.2f} мс, доля шума в её дисперсии {m['fast_noise_share']:.2f}")
    c = C['sd_slow']; print(f"  СКО медленной при наклоне: {c['rest']:.2f} → {c['tilt']:.2f}, рост у {c['up']}/{c['n']}, p = {c['p']:.4f}")
    print("\n4. Связь PR с предшествующим RR (медианы r; p — по записям, критерий Уилкоксона)")
    print(f"  вся запись (как в прежней версии): исходные {W_['r_raw']:+.2f}, сглаженные {W_['r_slow']:+.2f} "
          f"(положительна у {W_['r_slow_pos']}/{W_['n']}, p = {W_['r_slow_p']:.4f})")
    print(f"  вся запись после вычитания средних каждой фазы: сглаженные {W_['r_slow_centered']:+.2f} (p = {W_['r_cent_p']:.4f})")
    for ph in ('покой', 'наклон'):
        m = M[ph]; print(f"  {ph}: исходные {m['r_raw']:+.2f} (p {m['r_raw_p']:.3f}); медленные {m['r_slow']:+.2f} "
                         f"(>0 у {m['r_slow_pos']}/{m['n']}, p {m['r_slow_p']:.4f}; суррогатами значима у {m['surr_sig']}/{m['n']}); "
                         f"быстрые с предшествующим RR {m['r_fast_prev']:+.3f} (>0 у {m['r_fast_prev_pos']}/{m['n']}, p {m['r_fast_prev_p']:.4f}), "
                         f"с последующим {m['r_fast_next']:+.3f} (p {m['r_fast_next_p']:.4f})")
    for ph in ('покой', 'наклон'):
        m = M[ph]; print(f"  {ph}: r медленных с поправкой на шум {m['r_slow_corr']:+.2f}")
    ra, ta = collect(main, 'r_slow', 'покой'), collect(main, 'r_slow', 'наклон')
    print(f"  r медленных по записям: покой от {np.nanmin(ra):+.2f} до {np.nanmax(ra):+.2f}; отрицателен при наклоне у {int(np.nansum(ta < 0))}/{int(np.isfinite(ta).sum())}, "
          f"в обеих фазах у {int(np.sum((ra < 0) & (ta < 0)))} записей")
    for k, lab in (('r_slow', 'r медленных'), ('r_slow_corr', 'r медленных с поправкой на шум'), ('r_raw', 'r исходных'),
                   ('r_fast_prev', 'r быстрых с предшествующим RR'), ('r_fast_next', 'r быстрых с последующим RR')):
        c = C[k]; print(f"  покой → наклон, {lab}: {c['rest']:+.2f} → {c['tilt']:+.2f}, рост у {c['up']}/{c['n']}, p = {c['p']:.4f}")
    print("\n5. Средний уровень PR и зубец P по фазам")
    c = C['mean_pr']; print(f"  средний PR: {c['rest']:.1f} → {c['tilt']:.1f} мс, рост у {c['up']}/{c['n']}, p = {c['p']:.4f}")
    amp_r, amp_t, shape, off_r, off_t = [], [], [], [], []
    for n, p in zip(names, paths):
        v = S2[n]; Wn = phase_windows(v)
        ar, tr, orr = p_template(p, LEAD_II_CHANNEL, v['p_ref'], v['t_r'], Wn['покой']); at, tt, ot = p_template(p, LEAD_II_CHANNEL, v['p_ref'], v['t_r'], Wn['наклон'])
        amp_r.append(ar); amp_t.append(at); shape.append(corr(tr, tt) if tr is not None and tt is not None else np.nan); off_r.append(orr); off_t.append(ot)
    amp_r, amp_t, shape, off_r, off_t = map(lambda x: np.array(x, float), (amp_r, amp_t, shape, off_r, off_t))
    mpr = {s['rec']: s['phase'] for s in main}
    dpr = np.array([mpr[n]['наклон']['mean_pr'] - mpr[n]['покой']['mean_pr'] if mpr[n]['покой'] and mpr[n]['наклон'] else np.nan for n in names])
    okk = np.isfinite(amp_r) & np.isfinite(amp_t)
    summ['зубец_P'] = dict(amp_rest=med(amp_r), amp_tilt=med(amp_t), up=int(np.sum(amp_t[okk] > amp_r[okk])), n=int(okk.sum()),
                           p=wilcox(amp_t[okk] - amp_r[okk]), shape_r=med(shape), shape_min=float(np.nanmin(shape)),
                           shape_lt09=int(np.nansum(shape < 0.9)), dpr_med=med(dpr), dpr_n=int(np.isfinite(dpr).sum()),
                           dpr_down=int(np.nansum(dpr < 0)))
    for j, key in ((0, 'вершина'), (1, 'центр')):
        doff = off_t[:, j] - off_r[:, j]; okd = np.isfinite(doff) & np.isfinite(dpr); dc = dpr[okd] - doff[okd]
        summ['зубец_P'][key] = dict(off_med=med(doff[okd]), off_absmax=float(np.max(np.abs(doff[okd]))), dpr_med=med(dc),
                                   down=int(np.sum(dc < 0)), n=int(okd.sum()), p=wilcox(dc))
    zp = summ['зубец_P']
    print(f"  размах шаблона P (ЭКГ 0,5–40 Гц): покой {zp['amp_rest']:.0f} → наклон {zp['amp_tilt']:.0f}, рост у {zp['up']}/{zp['n']}, p = {zp['p']:.4f}")
    print(f"  сходство формы P в покое и при наклоне: медиана r {zp['shape_r']:.3f}, минимум {zp['shape_min']:.3f}, r < 0,9 у {zp['shape_lt09']} записей")
    print(f"  изменение среднего PR (наклон − покой) по записям: медиана {zp['dpr_med']:+.2f} мс, укорочение у {zp['dpr_down']}/{zp['dpr_n']}")
    for key, lab in (('вершина', 'вершины шаблона P своей фазы'), ('центр', 'центра верхней половины шаблона P своей фазы')):
        q = zp[key]; print(f"  если отсчитывать P от {lab}: смещение точки (наклон − покой) медиана {q['off_med']:+.2f} мс, "
                           f"наибольшее по модулю {q['off_absmax']:.2f} мс; изменение PR медиана {q['dpr_med']:+.2f} мс, "
                           f"укорочение у {q['down']}/{q['n']}, p = {q['p']:.2g}")
    print("\n6. Блоки циклов (СКО средних PR по блокам внутри фазы после удаления тренда, мс; медианы)")
    for ph in ('покой', 'наклон'):
        m = M[ph]
        print('  ' + ph + ': ' + '; '.join(f"{L} циклов — {m[f'block{L}']:.2f} (без медленных изменений {m[f'block{L}_fast']:.2f}, "
                                          f"от погрешности {m[f'block{L}_noise']:.2f}; больше у {m[f'block{L}_exceed']}/{m['n']})" for L in BLOCKS))
    print("\n7. Синтетическая проверка метода (искусственный PR на настоящих рядах RR)", flush=True)
    syn = synthetic_check(S2, {s['rec']: s['noise_cl_whole'] for s in main}); summ['синтетика'] = syn
    for sc, lab in (('а', 'медленная часть PR не связана с RR, сдвиг среднего при смене позы как в записи'),
                    ('б', 'медленная часть PR внутри фаз следует за RR на всех медленных масштабах'),
                    ('в', 'PR следует за RR только через медленный дрейф (больше ~100 циклов)')):
        s = syn[sc]; print(f"  ({sc}) {lab} [{s['n']} рядов]: вся запись {s['whole']:+.2f}, после вычитания средних фаз {s['centered']:+.2f}, "
                           f"покой {s['rest']:+.2f}, наклон {s['tilt']:+.2f}; после удаления медленного тренда (101 цикл): покой {s['rest101']:+.2f}, "
                           f"наклон {s['tilt101']:+.2f}; суррогатами значима (p < 0,05) в {s['detect_rest']:.0%} / {s['detect_tilt']:.0%} случаев")
    print("\n8. Варианты расчёта", flush=True)
    Sb = B(lambda n: P7_LEADS[n]); S2raw = B(lambda n: LEAD_II_CHANNEL, refine=False); S3raw = B(lambda n: LEAD_III_CHANNEL, refine=False)
    variants = {}
    for lab, kw, SS, S33 in (('основной расчёт', {}, S2, S3), ('сглаживание по 9 циклам', dict(smooth=9), S2, S3),
                             ('сглаживание по 25 циклам', dict(smooth=25), S2, S3), ('без удаления тренда', dict(detrend=False), S2, S3),
                             ('медленный тренд — среднее по 101 циклу', dict(detrend='ma101'), S2, S3),
                             ('лучшее отведение записи', {}, Sb, None), ('без уточнения положений R и P', {}, S2raw, S3raw)):
        st = [record_stats(n, SS[n], S33[n] if S33 else None, np.random.default_rng(SEED), with_noise=S33 is not None, **kw) for n in names]
        sm = summarize(st); c = sm['изменение']['r_slow']
        if lab.startswith('медленный тренд'):
            stats_101 = st
        cc = sm['изменение']['r_slow_corr']
        variants[lab] = dict(whole=sm['вся запись']['r_slow'], rest=sm['покой']['r_slow'], tilt=sm['наклон']['r_slow'], n=c['n'], up=c['up'], p=c['p'],
                             p_corr=cc['p'], up_corr=cc['up'],
                             rest_p=sm['покой']['r_slow_p'], tilt_p=sm['наклон']['r_slow_p'],
                             own_rest=sm['покой'].get('own_dec'), own_tilt=sm['наклон'].get('own_dec'),
                             noise_rest=sm['покой'].get('noise_dec'), noise_tilt=sm['наклон'].get('noise_dec'))
        v = variants[lab]
        print(f"  {lab:32s} r медленных: вся {v['whole']:+.2f} | покой {v['rest']:+.2f} (p {v['rest_p']:.3f}) | наклон {v['tilt']:+.2f} (p {v['tilt_p']:.4f}) | "
              f"рост {v['up']}/{v['n']}, p = {v['p']:.4f}" + (f"; с поправкой на шум рост {v['up_corr']}/{v['n']}, p = {v['p_corr']:.4f}" if np.isfinite(v['p_corr']) else ''), flush=True)
    summ['варианты'] = variants
    # файлы
    with open(os.path.join(out_dir, 'p0_per_record.csv'), 'w', newline='', encoding='utf-8-sig') as f:
        wr = csv.writer(f, delimiter=';')
        keys = ['sd_rr', 'sd_disc', 'noise_dec', 'noise_cl', 'own_dec', 'own_cl', 'diff_noise_share_cl', 'diff_noise_share_dec', 'ratio_diff', 'sd_slow', 'sd_fast', 'r_raw', 'r_slow', 'r_slow_corr', 'p_slow_surr',
                'r_fast_prev', 'r_fast_next', 'mean_pr', 'block20_sd', 'block40_sd', 'block80_sd']
        wr.writerow(['запись', 'фаза', 'охват PR'] + keys + ['r по всей записи (сглаж.)', 'r по всей записи после вычитания средних фаз'])
        for s in main:
            for ph in ('покой', 'наклон'):
                st = s['phase'][ph]
                wr.writerow([s['rec'], ph, f"{s['cover'][ph]:.3f}"] + ([f"{st[k]:.4f}" if st and np.isfinite(st[k]) else '' for k in keys] if st else [''] * len(keys))
                            + [f"{s['whole_r_slow']:.4f}", f"{s['whole_r_slow_centered']:.4f}"])
    with open(os.path.join(out_dir, 'p0_summary.json'), 'w', encoding='utf-8') as f:
        json.dump(_clean(summ), f, ensure_ascii=False, indent=1)
    fig_example(S2['0029'] if '0029' in S2 else S2[names[0]], '0029' if '0029' in S2 else names[0], os.path.join(out_dir, 'p0_fig1_example.png'))
    fig_rest_tilt(main, stats_101, os.path.join(out_dir, 'p0_fig3_rest_tilt.png'))
    fig_blocks(M, os.path.join(out_dir, 'p0_fig2_blocks.png'))
    print(f'\nРезультаты сохранены в папку {out_dir}')
    return summ


def _in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except Exception:
        return False


if __name__ == '__main__' and not _in_notebook():
    ap = argparse.ArgumentParser(description='Пункт 0: микровариативность интервала PR')
    ap.add_argument('data_dir'); ap.add_argument('--out', default='results_p0')
    a = ap.parse_args(); run_p0(a.data_dir, a.out)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 19.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyedflib: filename=pyedflib-0.1.42-cp313-cp313-linux_x86_64.whl size=2852054 sha256=ba2c35d37f08a123c98879ea5dc2f0c2d8db37e820d92843ae6e84b70dd54476
  Stored in directory: /root/.cache/pip/wheels/ac/16/3e/5299c8a6b023bfd3d7204240ca5de940a49bede1edb30015ab
Successfully built pyedflib


In [2]:
run_p0('/content/data/')

Записей: 20. Строю ряды (отведения II и III, лучшее отведение, вариант без уточнения)...

Охват PR по фазам (доля циклов): ниже 80% — 0018 покой (68%; зубец P найден в 74% циклов, разница — исключённые выбросы)

1. Расхождение PP и RR (медианы)
  покой: СКО RR 55.2 мс, СКО (PP − RR) 2.86 мс (5.2% от СКО RR), r(PP, RR) 0.9986
  наклон: СКО RR 62.6 мс, СКО (PP − RR) 2.19 мс (3.5% от СКО RR), r(PP, RR) 0.9992
  СКО (PP − RR) при наклоне (записи с обеими фазами): 2.86 → 2.16 мс, рост у 4/19, p = 0.0141

2. Погрешность и собственная изменчивость PR (медианы, мс)
  вся запись: погрешность — разложение 1.44, разница II–III 1.45; собственная 1.58
  погрешность, разложение: покой 1.54, наклон 1.12 (записей с обеими фазами 14); при наклоне больше у 2/14, p = 0.0052
  погрешность, разница II–III: покой 1.96, наклон 0.81 (записей с обеими фазами 19); при наклоне больше у 4/19, p = 0.0062
  собственная изменчивость разностей, разложение: покой 1.34, наклон 1.17 (записей с обеими фазами 14); при нак

{'основной': {'покой': {'sd_rr': 55.20511625422737,
   'sd_disc': 2.8613860125812693,
   'r_pp_rr': 0.9986271443933097,
   'noise_dec': 1.5273798309817175,
   'noise_cl': 1.9614660886011082,
   'own_dec': 1.4245498709518187,
   'own_cl': 1.4239369598466156,
   'r_jitter': 0.10553925623185316,
   'sd_pr': 3.2061425556838827,
   'sd_dpr': 2.8794336583780953,
   'ratio_diff': 1.0512132667746796,
   'sd_slow': 1.6169004453190226,
   'sd_fast': 2.0604375970878093,
   'r_raw': 0.08586354267627812,
   'r_slow': -0.3376134019365715,
   'r_slow_corr': -0.36845421434916004,
   'slow_noise_share': 0.16416479595633127,
   'r_fast_prev': 0.24514804534163584,
   'r_fast_next': 0.10946000306818295,
   'mean_pr': 146.23511975186,
   'fast_noise_share': 0.5570419582919276,
   'n': 19,
   'n_dec': 17,
   'own_gt_noise_dec': 8,
   'own_gt_noise_cl': 9,
   'n_cl': 19,
   'slow_vs_noise': 2.468084271553675,
   'r_raw_pos': 11,
   'r_raw_p': 0.6507492065429688,
   'r_slow_pos': 7,
   'r_slow_p': 0.156402587